# NeuraSight — Brain MRI Scan Training (All Models in One Run)

Trains **all four base models in a single run** for the stacking-ensemble research:
**EfficientNet-B0, ResNet-50, VGG-16, DenseNet-121**.

Each model saves to Drive the moment it finishes, so completed models are safe
even if the Colab session disconnects. For each model this notebook saves:
weights `BRAIN_MRI_<MODEL>.pth`, test-set probabilities `*_test_probs.npy`,
and a metrics summary `*_metrics.json` — used by the stacking meta-learner.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile

zip_path = "/content/drive/MyDrive/NeuraSight/brainMRI.zip"
extract_path = "/content/brainMRI"

print("Dataset zip exists:", os.path.exists(zip_path))

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully")

In [ ]:
!pip install timm grad-cam seaborn -q

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import timm
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from sklearn.metrics import (confusion_matrix, classification_report,
                             precision_recall_fscore_support, accuracy_score)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Configuration

`MODELS_TO_TRAIN` lists every model trained in this run. Per-model
hyperparameters (epochs, batch size, LR, optimizer, weight decay) are in
`HYPERPARAMS`. Note VGG-16 uses a lower learning rate.

In [ ]:
MODEL_CONFIG = {
    "efficientnet": {"timm_name": "efficientnet_b0", "save_name": "BRAIN_MRI_EFFICIENTNET"},
    "resnet":       {"timm_name": "resnet50",        "save_name": "BRAIN_MRI_RESNET"},
    "vgg":          {"timm_name": "vgg16",           "save_name": "BRAIN_MRI_VGG"},
    "densenet":     {"timm_name": "densenet121",     "save_name": "BRAIN_MRI_DENSENET"},
}

# Train all four in one run. Remove any you have already trained to resume.
MODELS_TO_TRAIN = ["efficientnet", "resnet", "densenet", "vgg"]

HYPERPARAMS = {
    "efficientnet": {"epochs": 15, "batch_size": 32, "lr": 1e-4, "weight_decay": 1e-2, "optimizer": "adamw"},
    "resnet":       {"epochs": 15, "batch_size": 32, "lr": 1e-4, "weight_decay": 1e-2, "optimizer": "adamw"},
    "vgg":          {"epochs": 15, "batch_size": 32, "lr": 1e-5, "weight_decay": 1e-2, "optimizer": "adamw"},
    "densenet":     {"epochs": 15, "batch_size": 32, "lr": 1e-4, "weight_decay": 1e-2, "optimizer": "adamw"},
}

SAVE_DIR = "/content/drive/MyDrive/NeuraSight/models"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Models to train:", MODELS_TO_TRAIN)
print("Saving to:", SAVE_DIR)

In [ ]:
train_dir = "/content/brainMRI/Training"
test_dir  = "/content/brainMRI/Testing"

print("TRAINING DATASET")
for cls in sorted(os.listdir(train_dir)):
    print(f"  {cls}: {len(os.listdir(os.path.join(train_dir, cls)))}")

print("TESTING DATASET")
for cls in sorted(os.listdir(test_dir)):
    print(f"  {cls}: {len(os.listdir(os.path.join(test_dir, cls)))}")

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
train_dataset = ImageFolder(train_dir, transform=train_transform)
test_dataset  = ImageFolder(test_dir,  transform=test_transform)

CLASS_NAMES = train_dataset.classes
print("Class order (index -> label):")
for i, c in enumerate(CLASS_NAMES):
    print(f"  {i}: {c}")
print("Train images:", len(train_dataset), "| Test images:", len(test_dataset))

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / len(loader), 100 * correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return running_loss / len(loader), 100 * correct / total

In [ ]:
def run_model(key):
    cfg = MODEL_CONFIG[key]
    hp = HYPERPARAMS[key]
    save_name = cfg["save_name"]
    best_path = os.path.join(SAVE_DIR, save_name + ".pth")
    print(f"Model: {key} ({cfg['timm_name']}) | HP: {hp}")

    train_loader = DataLoader(train_dataset, batch_size=hp["batch_size"], shuffle=True,
                              num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=hp["batch_size"], shuffle=False,
                             num_workers=2, pin_memory=True)

    model = timm.create_model(cfg["timm_name"], pretrained=True, num_classes=4).to(device)
    criterion = nn.CrossEntropyLoss()

    opt = hp["optimizer"].lower()
    if opt == "adamw":
        optimizer = optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    elif opt == "adam":
        optimizer = optim.Adam(model.parameters(), lr=hp["lr"])
    else:
        optimizer = optim.SGD(model.parameters(), lr=hp["lr"], momentum=0.9,
                              weight_decay=hp["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=hp["epochs"], eta_min=1e-6)

    best_acc = 0.0
    for epoch in range(hp["epochs"]):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        vl_loss, vl_acc = evaluate(model, test_loader, criterion, device)
        scheduler.step()
        print(f"  Epoch {epoch+1}/{hp['epochs']} | Train {tr_acc:.2f}% | Val {vl_acc:.2f}%")
        if vl_acc > best_acc:
            best_acc = vl_acc
            torch.save(model.state_dict(), best_path)
    print(f"  Best Val Acc: {best_acc:.2f}%  ->  saved {save_name}.pth")

    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()
    probs_list, y_true, y_pred = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images.to(device))
            probs = torch.softmax(outputs, dim=1)
            probs_list.append(probs.cpu().numpy())
            y_pred.extend(probs.argmax(1).cpu().numpy())
            y_true.extend(labels.numpy())
    probs = np.concatenate(probs_list, axis=0)
    y_true, y_pred = np.array(y_true), np.array(y_pred)

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(f"{key} Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.show()
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

    np.save(os.path.join(SAVE_DIR, save_name + "_test_probs.npy"), probs)
    np.save(os.path.join(SAVE_DIR, "test_labels.npy"), y_true)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro")
    metrics = {"model": key, "timm_name": cfg["timm_name"],
               "accuracy": round(accuracy_score(y_true, y_pred) * 100, 2),
               "precision": round(prec * 100, 2), "recall": round(rec * 100, 2),
               "f1": round(f1 * 100, 2), "best_val_acc": round(best_acc, 2)}
    with open(os.path.join(SAVE_DIR, save_name + "_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)
    return metrics

## Train All Models

Runs every model in `MODELS_TO_TRAIN` sequentially. Each finished model is
saved to Drive immediately.

In [ ]:
all_metrics = []
for key in MODELS_TO_TRAIN:
    print("=" * 60)
    print("TRAINING:", key)
    print("=" * 60)
    all_metrics.append(run_model(key))
    print()

print("All requested models trained.")

## Model Comparison Summary

In [ ]:
df = pd.DataFrame(all_metrics)
df = df[["model", "timm_name", "accuracy", "precision", "recall", "f1", "best_val_acc"]]
df = df.sort_values("accuracy", ascending=False).reset_index(drop=True)
df.to_csv(os.path.join(SAVE_DIR, "model_comparison.csv"), index=False)
print(df.to_string(index=False))

In [ ]:
plt.figure(figsize=(9, 5))
metrics_to_plot = ["accuracy", "precision", "recall", "f1"]
x = np.arange(len(df))
width = 0.2
for i, m in enumerate(metrics_to_plot):
    plt.bar(x + i * width, df[m], width, label=m.capitalize())
plt.xticks(x + 1.5 * width, df["model"])
plt.ylabel("Score (%)"); plt.title("Base Model Comparison")
plt.ylim(80, 100); plt.legend()
plt.show()

In [ ]:
from google.colab import files
print("Artifacts in", SAVE_DIR, ":")
for fn in sorted(os.listdir(SAVE_DIR)):
    print("  ", fn)

# Download all trained weights to your local machine
for key in MODELS_TO_TRAIN:
    p = os.path.join(SAVE_DIR, MODEL_CONFIG[key]["save_name"] + ".pth")
    if os.path.exists(p):
        files.download(p)

## Next Steps

After this run, `Drive/NeuraSight/models/` contains all four
`BRAIN_MRI_*.pth` weights, their `*_test_probs.npy`, `*_metrics.json`,
`test_labels.npy`, and `model_comparison.csv`.

Next: **notebook 03 (meta-learner)** trains the Logistic Regression stacking
ensemble on the saved probabilities (with a leakage-safe split).